# Notebook 02 — GPS data preprocessing

This notebook preprocesses the existing `LBBG_SCHIERMONNIKOOG-gps-2017.csv.gz` file for reproducible analysis. The source file is read without modification and all source columns are retained. The records are GPS fixes, not movement steps; this notebook does not calculate distances, infer stopovers, remove flagged records, or interpret UTC as local solar time.

Dataset: LBBG_SCHIERMONNIKOOG, Zenodo DOI `10.5281/zenodo.20706447`, CC0. The canonical analytical table contains every source row and adds documented canonical and UTC-derived fields.

## Reproducibility and provenance

Paths are resolved from the repository working directory. The input is discovered locally under `data/raw`; no download occurs. Package versions, input file metadata, transformation counts, QC reports, figures, and the Parquet output are recorded by the executed cells below. Critical validation failures raise an exception.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow

PROJECT_ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / 'data' / 'raw').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Could not locate repository root from working directory: {Path.cwd()}')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
for directory in (REPORT_DIR, FIGURE_DIR, PROCESSED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

candidates = sorted(RAW_DIR.glob('LBBG_SCHIERMONNIKOOG-gps-*.csv.gz'))
if len(candidates) != 1:
    raise FileNotFoundError(f'Expected exactly one local compressed GPS CSV, found: {candidates}')
INPUT_PATH = candidates[0]
PARQUET_PATH = PROCESSED_DIR / 'lbbg_gps_fixes_preprocessed.parquet'
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('matplotlib:', matplotlib.__version__)
print('pyarrow:', pyarrow.__version__)
print('Input:', INPUT_PATH)
print('Input bytes:', INPUT_PATH.stat().st_size)


Python: 3.12.12
Platform: macOS-15.6-arm64-arm-64bit
pandas: 2.3.3
numpy: 2.5.4
matplotlib: 3.11.2
pyarrow: 19.0.1
Input: /Users/rizqititing/Documents/projects/bird-migration/data/raw/LBBG_SCHIERMONNIKOOG-gps-2017.csv.gz
Input bytes: 22632540


In [2]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

INPUT_SHA256 = sha256(INPUT_PATH)
raw_df = pd.read_csv(INPUT_PATH, compression='gzip', low_memory=False)
source_columns = raw_df.columns.tolist()
source_row_count = len(raw_df)
print(f'Loaded {source_row_count:,} GPS fixes and {len(source_columns)} source columns')
print('SHA-256:', INPUT_SHA256)
display(raw_df.head(3))


Loaded 441,979 GPS fixes and 22 source columns
SHA-256: da41dcaf3048075049b11dc77033f5911e00ec431a4d673f144619cff1c69d8c


,event-id,visible,timestamp,location-long,location-lat,bar:barometric-pressure,external-temperature,gps:dop,gps:satellite-count,gps-time-to-fix,...,height-above-msl,import-marked-outlier,location-error-numerical,manually-marked-outlier,vertical-error-numerical,sensor-type,individual-taxon-canonical-name,tag-local-identifier,individual-local-identifier,study-name
0,50219527862,True,2017-05-24 09:46:47.000,6.262858,53.499042,NaN,20.3,5.2,5,20.0,...,106.0,False,6.8,False,6.1,gps,Larus fuscus,5527,5500276,LBBG_SCHIERMONNIKOOG - Lesser black-backed gul...
1,50219527863,True,2017-05-24 09:56:43.000,6.267735,53.506315,NaN,28.5,5.0,5,8.2,...,-6.0,False,11.8,False,8.1,gps,Larus fuscus,5527,5500276,LBBG_SCHIERMONNIKOOG - Lesser black-backed gul...
2,50219527864,True,2017-05-24 10:07:09.000,6.267793,53.506317,NaN,29.1,1.7,7,26.2,...,0.0,False,4.8,False,4.2,gps,Larus fuscus,5527,5500276,LBBG_SCHIERMONNIKOOG - Lesser black-backed gul...


## Canonical fields and UTC temporal fields

The original source fields remain in the table. Canonical fields are additive aliases or parsed representations: `event_id`, `individual_id`, `timestamp_utc`, `longitude`, `latitude`, `sensor_type`, `location_error_m`, `ground_speed_source`, and `height_above_msl`. UTC calendar fields are derived only from the documented UTC/GPS timestamp; no local solar-time conversion is made.

In [3]:
df = raw_df.copy(deep=True)
canonical_source_map = {
    'event_id': 'event-id',
    'individual_id': 'individual-local-identifier',
    'timestamp_utc': 'timestamp',
    'longitude': 'location-long',
    'latitude': 'location-lat',
    'sensor_type': 'sensor-type',
    'location_error_m': 'location-error-numerical',
    'ground_speed_source': 'ground-speed',
    'height_above_msl': 'height-above-msl',
}
missing_mappings = set(canonical_source_map.values()) - set(df.columns)
if missing_mappings:
    raise KeyError(f'Required source columns missing: {sorted(missing_mappings)}')
pd.DataFrame([{'canonical_field': canonical, 'source_column': source} for canonical, source in canonical_source_map.items()]).to_csv(REPORT_DIR / 'preprocessing_column_mapping.csv', index=False)

df['event_id'] = df['event-id'].astype('string')
df['individual_id'] = df['individual-local-identifier'].astype('string')
df['timestamp_utc'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d %H:%M:%S.%f', utc=True, errors='coerce')
df['longitude'] = pd.to_numeric(df['location-long'], errors='coerce')
df['latitude'] = pd.to_numeric(df['location-lat'], errors='coerce')
df['sensor_type'] = df['sensor-type'].astype('string')
df['location_error_m'] = pd.to_numeric(df['location-error-numerical'], errors='coerce')
df['ground_speed_source'] = pd.to_numeric(df['ground-speed'], errors='coerce')
df['height_above_msl'] = pd.to_numeric(df['height-above-msl'], errors='coerce')

df['utc_date'] = df['timestamp_utc'].dt.date.astype('string')
df['utc_year'] = df['timestamp_utc'].dt.year.astype('Int64')
df['utc_month'] = df['timestamp_utc'].dt.month.astype('Int64')
df['utc_day'] = df['timestamp_utc'].dt.day.astype('Int64')
df['utc_hour'] = df['timestamp_utc'].dt.hour.astype('Int64')
df['utc_dayofweek'] = df['timestamp_utc'].dt.dayofweek.astype('Int64')
df['utc_dayofyear'] = df['timestamp_utc'].dt.dayofyear.astype('Int64')

df = df.sort_values(['individual_id', 'timestamp_utc', 'event_id'], kind='mergesort').reset_index(drop=True)
print('Canonical and derived fields added:', len(df.columns) - len(source_columns))
print('Output columns:', len(df.columns))


Canonical and derived fields added: 16
Output columns: 38


## Critical validation

These checks validate the source audit and the canonical representations. No rows are excluded. Missing speed/error values and outlier-flagged records remain in the analytical DataFrame.

In [4]:
timestamp_failures = int(df['timestamp_utc'].isna().sum())
invalid_coordinates = int((df['latitude'].isna() | df['longitude'].isna() | ~df['latitude'].between(-90, 90) | ~df['longitude'].between(-180, 180)).sum())
missing_ids = int(df['individual_id'].isna().sum())
duplicate_event_ids = int(df['event_id'].duplicated().sum())
critical_failures = {
    'row_count_changed': len(df) != source_row_count,
    'timestamp_parse_failures': timestamp_failures,
    'invalid_coordinates': invalid_coordinates,
    'missing_individual_ids': missing_ids,
    'duplicate_event_ids': duplicate_event_ids,
}
if any(critical_failures.values()):
    raise ValueError(f'Critical preprocessing validation failed: {critical_failures}')
assert len(df) == 441_979, 'Unexpected row count relative to Notebook 01 audit'
assert df['individual_id'].nunique() == 20, 'Unexpected individual count relative to Notebook 01 audit'
assert df['timestamp_utc'].min() == pd.Timestamp('2017-05-24 09:46:47', tz='UTC')
assert df['timestamp_utc'].max() == pd.Timestamp('2017-12-31 23:58:28', tz='UTC')
print('Critical validation passed:', critical_failures)
print('Individuals:', df['individual_id'].nunique())
print('UTC range:', df['timestamp_utc'].min(), 'to', df['timestamp_utc'].max())


Critical validation passed: {'row_count_changed': False, 'timestamp_parse_failures': 0, 'invalid_coordinates': 0, 'missing_individual_ids': 0, 'duplicate_event_ids': 0}
Individuals: 20
UTC range: 2017-05-24 09:46:47+00:00 to 2017-12-31 23:58:28+00:00


In [5]:
missingness = (
    df[source_columns]
    .isna()
    .agg(['sum', 'mean'])
    .T
    .rename(columns={'sum': 'missing_count', 'mean': 'missing_fraction'})
    .assign(column=lambda x: x.index, dtype=lambda x: df[x.name].dtype if False else None)
)
missingness = pd.DataFrame({
    'column': source_columns,
    'missing_count': [int(df[c].isna().sum()) for c in source_columns],
    'missing_fraction': [float(df[c].isna().mean()) for c in source_columns],
    'dtype': [str(df[c].dtype) for c in source_columns],
}).sort_values(['missing_count', 'column'], ascending=[False, True])
missingness.to_csv(REPORT_DIR / 'preprocessing_missingness.csv', index=False)
display(missingness.head(10))

outlier_columns = ['visible', 'import-marked-outlier', 'manually-marked-outlier']
outlier_rows = []
for column in outlier_columns:
    counts = df[column].value_counts(dropna=False)
    for value, count in counts.items():
        outlier_rows.append({'field': column, 'value': str(value), 'count': int(count), 'fraction': float(count / len(df))})
outlier_summary = pd.DataFrame(outlier_rows).sort_values(['field', 'value'])
outlier_summary.to_csv(REPORT_DIR / 'preprocessing_outliers.csv', index=False)
display(outlier_summary)


,column,missing_count,missing_fraction,dtype
5,bar:barometric-pressure,441979,1.000000,float64
10,ground-speed,2526,0.005715,float64
11,heading,2526,0.005715,float64
14,location-error-numerical,2526,0.005715,float64
16,vertical-error-numerical,2526,0.005715,float64
0,event-id,0,0.000000,int64
6,external-temperature,0,0.000000,float64
9,gps-time-to-fix,0,0.000000,float64
7,gps:dop,0,0.000000,float64
8,gps:satellite-count,0,0.000000,int64


,field,value,count,fraction
1,import-marked-outlier,False,441979,1.0
2,manually-marked-outlier,False,441979,1.0
0,visible,True,441979,1.0


## QC summaries and figures

The plots describe observation coverage and quality flags. They are descriptive only; flagged records are not filtered. Daily counts and coverage are based on UTC dates.

In [ ]:
individual_summary = (
    df.groupby('individual_id', dropna=False)
    .agg(fix_count=('event_id', 'size'), first_timestamp_utc=('timestamp_utc', 'min'), last_timestamp_utc=('timestamp_utc', 'max'),
         latitude_min=('latitude', 'min'), latitude_max=('latitude', 'max'), longitude_min=('longitude', 'min'), longitude_max=('longitude', 'max'))
    .reset_index()
)
individual_summary['coverage_days'] = (individual_summary['last_timestamp_utc'] - individual_summary['first_timestamp_utc']).dt.total_seconds() / 86400
individual_summary.to_csv(REPORT_DIR / 'preprocessing_individual_summary.csv', index=False)

daily_fixes = df.groupby('utc_date', as_index=False).size().rename(columns={'size': 'fix_count'})
daily_fixes.to_csv(REPORT_DIR / 'preprocessing_daily_fixes.csv', index=False)

coverage = (df.groupby('individual_id', as_index=False).agg(first_timestamp_utc=('timestamp_utc', 'min'), last_timestamp_utc=('timestamp_utc', 'max'), fix_count=('event_id', 'size')))
coverage.to_csv(REPORT_DIR / 'preprocessing_tracking_coverage.csv', index=False)

plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(9, 7))
individual_summary.sort_values('fix_count').plot.barh(x='individual_id', y='fix_count', ax=ax, legend=False, color='#35618d')
ax.set(title='GPS fixes per individual', xlabel='Number of fixes', ylabel='Individual ID')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '02_fixes_per_individual.png', dpi=160); plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pd.to_datetime(daily_fixes['utc_date']), daily_fixes['fix_count'], color='#2a7f62', linewidth=1)
ax.set(title='Daily GPS fixes (UTC date)', xlabel='UTC date', ylabel='Fix count')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '02_daily_fixes.png', dpi=160); plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 7))
plot_missing = missingness[missingness['missing_count'] > 0].sort_values('missing_fraction')
ax.barh(plot_missing['column'], plot_missing['missing_fraction'], color='#c4772f')
ax.set(title='Missing source values', xlabel='Fraction missing', ylabel='Source column')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '02_missingness.png', dpi=160); plt.close(fig)

flag_counts = pd.DataFrame({
    'import_marked_outlier': [int(df['import-marked-outlier'].sum())],
    'manually_marked_outlier': [int(df['manually-marked-outlier'].sum())],
    'not_visible': [int((~df['visible']).sum())],
}).T.reset_index().rename(columns={'index': 'flag', 0: 'count'})
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(flag_counts['flag'], flag_counts['count'], color='#9b3d5a')
ax.set(title='Source QC flags (records retained)', ylabel='Record count'); ax.tick_params(axis='x', rotation=20)
fig.tight_layout(); fig.savefig(FIGURE_DIR / '02_outlier_flags.png', dpi=160); plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 7))
for individual_id, group in coverage.sort_values('first_timestamp_utc').reset_index(drop=True).iterrows():
    ax.plot([group['first_timestamp_utc'], group['last_timestamp_utc']], [individual_id, individual_id], linewidth=5, solid_capstyle='butt')
ax.set_yticks(range(len(coverage))); ax.set_yticklabels(coverage['individual_id'])
ax.set(title='Individual tracking coverage (UTC)', xlabel='Timestamp', ylabel='Individual ID')
fig.tight_layout(); fig.savefig(FIGURE_DIR / '02_tracking_coverage.png', dpi=160); plt.close(fig)
print('Saved QC tables and figures to', REPORT_DIR)



Saved QC tables and figures to /Users/rizqititing/Documents/projects/bird-migration/reports


## Analytical DataFrame and Parquet export

`analysis_df` is the documented analytical DataFrame. It contains all 441,979 rows, all 22 source columns, canonical aliases, and UTC temporal fields. No observations are removed because of missing speed/error values or source outlier flags.

In [7]:
analysis_df = df.copy()
excluded_rows = source_row_count - len(analysis_df)
if excluded_rows != 0:
    raise RuntimeError(f'Unexpected row exclusion: {excluded_rows}')
analysis_df.to_parquet(PARQUET_PATH, engine='pyarrow', index=False)

summary = pd.DataFrame([
    {'metric': 'input_file', 'value': str(INPUT_PATH)},
    {'metric': 'input_sha256', 'value': INPUT_SHA256},
    {'metric': 'input_rows', 'value': source_row_count},
    {'metric': 'processed_rows', 'value': len(analysis_df)},
    {'metric': 'excluded_rows', 'value': excluded_rows},
    {'metric': 'source_columns', 'value': len(source_columns)},
    {'metric': 'processed_columns', 'value': len(analysis_df.columns)},
    {'metric': 'individuals', 'value': analysis_df['individual_id'].nunique()},
    {'metric': 'outlier_import_marked_count', 'value': int(analysis_df['import-marked-outlier'].sum())},
    {'metric': 'outlier_manual_marked_count', 'value': int(analysis_df['manually-marked-outlier'].sum())},
    {'metric': 'not_visible_count', 'value': int((~analysis_df['visible']).sum())},
    {'metric': 'timestamp_min_utc', 'value': str(analysis_df['timestamp_utc'].min())},
    {'metric': 'timestamp_max_utc', 'value': str(analysis_df['timestamp_utc'].max())},
    {'metric': 'transformations', 'value': 'Added canonical aliases, parsed UTC timestamp, derived UTC calendar fields, sorted rows; retained all source rows.'},
])
summary.to_csv(REPORT_DIR / 'preprocessing_summary.csv', index=False)
print('Exported:', PARQUET_PATH)
print('Rows:', len(analysis_df), 'Columns:', len(analysis_df.columns), 'Individuals:', analysis_df['individual_id'].nunique())


Exported: /Users/rizqititing/Documents/projects/bird-migration/data/processed/lbbg_gps_fixes_preprocessed.parquet
Rows: 441979 Columns: 38 Individuals: 20


## Reload and final validation

The final check reloads the persisted Parquet file rather than validating only the in-memory object.

In [8]:
reloaded = pd.read_parquet(PARQUET_PATH, engine='pyarrow')
assert len(reloaded) == source_row_count
assert len(reloaded.columns) == len(analysis_df.columns)
assert set(reloaded['individual_id'].dropna().unique()) == set(analysis_df['individual_id'].dropna().unique())
assert reloaded['timestamp_utc'].min() == analysis_df['timestamp_utc'].min()
assert reloaded['timestamp_utc'].max() == analysis_df['timestamp_utc'].max()
assert reloaded['event_id'].is_unique
print('Parquet reload validation passed')
print('Rows:', len(reloaded))
print('Individuals:', reloaded['individual_id'].nunique())
print('Timestamp range:', reloaded['timestamp_utc'].min(), 'to', reloaded['timestamp_utc'].max())
print('Warnings: no rows excluded; source flags and missing values retained; UTC fields are not local solar time; movement distances not derived.')


Parquet reload validation passed
Rows: 441979
Individuals: 20
Timestamp range: 2017-05-24 09:46:47+00:00 to 2017-12-31 23:58:28+00:00
Warnings: no rows excluded; source flags and missing values retained; UTC fields are not local solar time; movement distances not derived.
